# Kandinsky V2A — Cat and Dog Event-Layer Personal Pilot

## Purpose

This is a one-off, private generation experiment using the local video file `12138626_1080_2320_30fps.mp4`. It compares one full-scene generation with separate candidate audio layers for the visible cat and dog.

This notebook does **not** train or fine-tune a model. The source video is not included in GitHub or in the downloaded results package.

## Experiment flow

1. Upload the video from your computer to this Colab runtime.
2. Inspect its media metadata and make a silent copy for visual analysis.
3. Ask Qwen for a timestamped map of visible cat/dog actions and a suggested five-second window.
4. Review the suggested window and prompts.
5. Run three Kandinsky Full Lite generations with the same five-second video and seed.
6. Compare the full-scene output with the cat and dog layers, then mix the two layers.

## Scope

Use this as a private, personal generation trial. It is not a training dataset or a public benchmark. If you later want to use Pexels footage for formal product evaluation, model training, or publication, check the exact clip’s terms and obtain permission where required. Pexels’ [Terms of Service](https://www.pexels.com/terms-of-service/) describe both content permissions and prohibited uses.

## 1 — Check the Colab runtime and prepare folders

Use an A100 runtime for the Full Lite checkpoint. The notebook will stop early with a clear message if a GPU or enough free disk space is unavailable.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, gc
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Audio, Video, display, Markdown

print("Python:", sys.version.split()[0])
if sys.version_info < (3, 13):
    raise RuntimeError("Use the Python 3.13+ Colab runtime used by the Kandinsky pilot.")

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU runtime is missing. Select an A100 runtime and reconnect.")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True
)
print("GPU:", gpu.stdout.strip())

roots = [Path("/content/local-scratch"), Path("/content")]
available_roots = [p for p in roots if p.exists()]
CACHE_ROOT = max(available_roots, key=lambda p: shutil.disk_usage(p).free)
free_gib = shutil.disk_usage(CACHE_ROOT).free / (1024 ** 3)
print(f"Cache root: {CACHE_ROOT} | free: {free_gib:.1f} GiB")
if free_gib < 35:
    raise RuntimeError("At least 35 GiB free disk space is recommended.")

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
os.environ["KANDINSKY_HOME"] = str(CACHE_ROOT / "kandinsky_home")

WORK = Path("/content/kandinsky_pet_event_layers_personal")
INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "outputs"
for folder in (INPUT_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, tail=3000):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-tail:])
    if result.returncode:
        if result.stderr:
            print(result.stderr[-6000:])
        raise RuntimeError("Command failed. See the output above.")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def show_video_preview(path):
    path = Path(path)
    if not path.is_file():
        print(f"Video preview skipped; file not found: {path}")
        return
    try:
        display(Video(data=path.read_bytes(), embed=True, mimetype="video/mp4"))
    except Exception as error:
        print(f"Video preview unavailable ({type(error).__name__}: {error}). File is present at: {path}")

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("This Colab runtime needs ffmpeg and ffprobe available on PATH.")

## 2 — Upload the video and inspect it

Colab runs on a remote machine and cannot read the Windows Downloads folder directly. Upload the single MP4 once in this cell. The video stays in this Colab runtime and is excluded from the results ZIP.

In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one MP4 file for this personal trial.")

filename, video_bytes = next(iter(uploaded.items()))
if Path(filename).suffix.lower() != ".mp4":
    raise ValueError(f"Expected an MP4 file, received: {filename}")

SOURCE_VIDEO = INPUT_DIR / Path(filename).name
SOURCE_VIDEO.write_bytes(video_bytes)
del uploaded, video_bytes

probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate,sample_rate,channels",
        "-of", "json", str(SOURCE_VIDEO)
    ],
    capture_output=True, text=True
)
if probe.returncode:
    raise RuntimeError("Could not inspect the uploaded video: " + probe.stderr[-2000:])

SOURCE_METADATA = json.loads(probe.stdout)
SOURCE_DURATION = float(SOURCE_METADATA["format"]["duration"])
SOURCE_SHA256 = sha256_file(SOURCE_VIDEO)
VIDEO_STREAMS = [s for s in SOURCE_METADATA.get("streams", []) if s.get("codec_type") == "video"]
AUDIO_STREAMS = [s for s in SOURCE_METADATA.get("streams", []) if s.get("codec_type") == "audio"]

print("File:", SOURCE_VIDEO.name)
print(f"Duration: {SOURCE_DURATION:.3f} seconds")
print("Video stream:", VIDEO_STREAMS[0] if VIDEO_STREAMS else "not found")
print("Audio stream:", AUDIO_STREAMS[0] if AUDIO_STREAMS else "not found")
print("SHA-256:", SOURCE_SHA256)

if SOURCE_DURATION < 5:
    raise RuntimeError("This Kandinsky pilot needs a source video that is at least five seconds long.")

## 3 — Make a silent analysis copy

The Qwen pass uses video frames only. We remove the source audio so it cannot influence the event map. This copy is downscaled and sampled at two frames per second to reduce analysis cost.

In [ ]:
SILENT_ANALYSIS_VIDEO = INPUT_DIR / "silent_analysis.mp4"
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(SOURCE_VIDEO),
    "-map", "0:v:0", "-an",
    "-vf", "fps=2,scale='min(640,iw)':-2",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "27",
    "-pix_fmt", "yuv420p", str(SILENT_ANALYSIS_VIDEO)
])
print("Silent analysis copy:", SILENT_ANALYSIS_VIDEO.name)
show_video_preview(SILENT_ANALYSIS_VIDEO)

## 4 — Ask Qwen for a visual event map

Qwen sees the silent video frames. It does not hear the source clip, so its result is a visual proposal, not a confirmed sound annotation. Review the timestamps and evidence before generation.

In [ ]:
import importlib.util

packages_needed = (
    importlib.util.find_spec("transformers") is None
    or importlib.util.find_spec("qwen_vl_utils") is None
    or importlib.util.find_spec("accelerate") is None
)
if packages_needed:
    run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.57.1", "accelerate", "qwen-vl-utils==0.0.14"
    ], tail=2000)

import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

QWEN_ID = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(QWEN_ID)
qwen = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

analysis_request = """Inspect this silent video and map visible actions involving the cat and dog.
Return valid JSON only:
{
  "events": [
    {
      "id": "E1",
      "subject": "cat | dog | both | other",
      "visible_action": "short description",
      "start_s": 0.0,
      "end_s": 1.0,
      "visual_evidence": "what is visible",
      "confidence": 0.0
    }
  ],
  "recommended_window_start_s": 0.0,
  "window_reason": "why this five-second window best shows the visible actions"
}
Use timestamps within the video duration. Recommend a five-second interval that best shows distinct cat and dog movement or interaction. Do not invent sounds, speech, music, materials, or off-screen events. If an action or its timing is uncertain, lower confidence."""

messages = [{
    "role": "user",
    "content": [
        {"type": "video", "video": SILENT_ANALYSIS_VIDEO.as_uri(), "fps": 2.0},
        {"type": "text", "text": analysis_request}
    ]
}]

qwen_text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
images, videos, video_kwargs = process_vision_info(
    messages, image_patch_size=16,
    return_video_kwargs=True, return_video_metadata=True
)
if videos is not None:
    videos, video_metadata = zip(*videos)
    videos, video_metadata = list(videos), list(video_metadata)
else:
    video_metadata = None

qwen_inputs = processor(
    text=qwen_text, images=images, videos=videos,
    video_metadata=video_metadata, padding=True,
    return_tensors="pt", do_resize=False, **video_kwargs
).to(qwen.device)

started = time.time()
qwen_ids = qwen.generate(**qwen_inputs, max_new_tokens=700, do_sample=False)
new_ids = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(qwen_inputs.input_ids, qwen_ids)
]
QWEN_EVENT_TEXT = processor.batch_decode(
    new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
QWEN_ANALYSIS_SECONDS = round(time.time() - started, 2)
print("Qwen analysis time:", QWEN_ANALYSIS_SECONDS, "seconds")
print(QWEN_EVENT_TEXT)

try:
    json_start, json_end = QWEN_EVENT_TEXT.find("{"), QWEN_EVENT_TEXT.rfind("}")
    if json_start < 0 or json_end <= json_start:
        raise ValueError("No JSON object found in Qwen response.")
    QWEN_PLAN = json.loads(QWEN_EVENT_TEXT[json_start:json_end + 1])
    print("\nParsed events:")
    print(json.dumps(QWEN_PLAN.get("events", []), ensure_ascii=False, indent=2))
except Exception as error:
    QWEN_PLAN = {"events": [], "recommended_window_start_s": 0.0}
    print("\nCould not parse the response automatically:", error)
    print("Use the raw Qwen response above and set the five-second window manually.")

## 5 — Review the event proposal and choose the five-second window

The notebook starts with Qwen’s suggested window. If it misses the cat/dog interaction, change `WINDOW_START_OVERRIDE` and rerun this cell. The preview lets you confirm the exact video segment before spending GPU time on Kandinsky.

In [ ]:
suggested_start = float(QWEN_PLAN.get("recommended_window_start_s", 0.0) or 0.0)
WINDOW_START_OVERRIDE = None  # Set a number of seconds here if the Qwen suggestion misses the action.
WINDOW_SECONDS = 5.0
max_start = max(0.0, SOURCE_DURATION - WINDOW_SECONDS)
WINDOW_START_SECONDS = suggested_start if WINDOW_START_OVERRIDE is None else float(WINDOW_START_OVERRIDE)
WINDOW_START_SECONDS = min(max(WINDOW_START_SECONDS, 0.0), max_start)

SILENT_WINDOW = INPUT_DIR / "cat_dog_window_5s.mp4"
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-ss", f"{WINDOW_START_SECONDS:.3f}", "-i", str(SOURCE_VIDEO),
    "-t", str(WINDOW_SECONDS), "-map", "0:v:0", "-an",
    "-vf", "fps=24,scale='min(960,iw)':-2",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
    "-pix_fmt", "yuv420p", str(SILENT_WINDOW)
])

print(f"Selected window: {WINDOW_START_SECONDS:.2f}–{WINDOW_START_SECONDS + WINDOW_SECONDS:.2f} seconds")
print("Review this exact five-second input before generation.")
show_video_preview(SILENT_WINDOW)

## 6 — Review and edit the three prompts

Kandinsky V2A currently needs both video and text. The prompts below deliberately ask for audible cat/dog vocalizations as a control: a clear cat meow and a short friendly dog woof. The silent video cannot tell us whether either animal actually vocalized in the original scene, so these calls are prompt-directed additions.

All three jobs receive the same five-second video, Full Lite checkpoint, and seed. Separate prompts do not guarantee isolated audio stems, so listen for vocalization leakage between cat and dog layers.

In [ ]:
# Edit these after reviewing the video and Qwen event map.
# This is a prompt-control trial: the vocalizations are intentionally requested,
# not inferred from the silent source video's original audio.
SEED = 42

PROMPTS = {
    "full_scene": (
        "A domestic cat makes one or two clear, natural meows. A nearby friendly dog "
        "answers with one or two short, soft woofs. Add subtle paw movement that follows "
        "the visible actions. No music or human speech."
    ),
    "cat_layer": (
        "Close foreground sound of a domestic cat: one or two clear, natural meows with "
        "subtle paw movement. Keep dog vocalizations very quiet. No music or human speech."
    ),
    "dog_layer": (
        "Close foreground sound of a friendly domestic dog: one or two short, soft, natural "
        "woofs with subtle paw movement. Keep cat vocalizations very quiet. No music or human speech."
    ),
}
print(json.dumps(PROMPTS, ensure_ascii=False, indent=2))

## 7 — Release Qwen and prepare the Kandinsky Full Lite runtime

Qwen is no longer needed. Free its GPU memory before loading Kandinsky. The Kandinsky source revision is pinned to the same one used by the previous Full Lite quality comparison.

In [ ]:
del qwen, processor, qwen_inputs, qwen_ids, images, videos, video_metadata
gc.collect()
torch.cuda.empty_cache()
print("Qwen released.")

REPO = Path("/content/kandinsky-6")
PINNED_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"

if not (REPO / ".git").exists():
    run([
        "git", "clone", "--filter=blob:none",
        "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)
    ], tail=2500)

run(["git", "-C", str(REPO), "checkout", PINNED_COMMIT], tail=1500)
run([sys.executable, "-m", "pip", "install", "-q", "uv"], tail=1500)

if not Path("/usr/local/bin/just").exists():
    run([
        "bash", "-lc",
        "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"
    ], tail=2000)

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
print(setup.stdout[-3500:])
if setup.returncode:
    print(setup.stderr[-6000:])
    raise RuntimeError("Kandinsky environment setup failed; inspect the output above.")

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Kandinsky source commit:", SOURCE_COMMIT)
run(["just", "download", "lite"], cwd=REPO, tail=2500)
print("Full Lite checkpoint is ready.")

## 8 — Generate one full-scene output and two candidate layers

The three calls run sequentially on one GPU. Each call uses the same silent video window and seed. The prompt is the only intended difference.

In [ ]:
DEVICE_CONFIG = REPO / "pet-event-layers-full-lite.yaml"
DEVICE_CONFIG.write_text(
    """checkpoint: lite
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""",
    encoding="utf-8"
)

GENERATION_JOBS = [
    {"id": key, "prompt": value}
    for key, value in PROMPTS.items()
]
CANDIDATES = []

for job in GENERATION_JOBS:
    output_mp4 = OUTPUT_DIR / f"{job['id']}.mp4"
    output_wav = OUTPUT_DIR / f"{job['id']}.wav"
    command = [
        "just", "generate", job["prompt"],
        "--config", str(DEVICE_CONFIG),
        "--attention-engine", "sdpa",
        "--video-file", str(SILENT_WINDOW),
        "--time-length", str(int(WINDOW_SECONDS)),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]

    print(f"Generating {job['id']} with Full Lite...")
    started = time.time()
    run(command, cwd=REPO, tail=3000)
    generation_seconds = round(time.time() - started, 2)

    if not output_mp4.is_file():
        raise FileNotFoundError(f"Generated MP4 is missing: {output_mp4}")

    run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100",
        "-c:a", "pcm_s16le", str(output_wav)
    ])
    if not output_wav.is_file():
        raise FileNotFoundError(f"Generated WAV is missing: {output_wav}")

    CANDIDATES.append({
        "id": job["id"],
        "prompt": job["prompt"],
        "seed": SEED,
        "generation_seconds": generation_seconds,
        "mp4": output_mp4,
        "wav": output_wav,
        "wav_sha256": sha256_file(output_wav),
    })
    print(f"Finished {job['id']} in {generation_seconds:.1f} seconds.")

## 9 — Mix the cat and dog candidates

This simple editable mix starts both layers together at equal level. If one layer contains the other animal or the whole scene, that leakage will remain audible and count as a result.

In [ ]:
CAT_WAV = OUTPUT_DIR / "cat_layer.wav"
DOG_WAV = OUTPUT_DIR / "dog_layer.wav"
LAYER_MIX_WAV = OUTPUT_DIR / "cat_dog_layer_mix.wav"

if not CAT_WAV.is_file() or not DOG_WAV.is_file():
    raise FileNotFoundError("Both animal-layer WAVs must exist before mixing.")

CAT_GAIN = 0.8
DOG_GAIN = 0.8
run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(CAT_WAV), "-i", str(DOG_WAV),
    "-filter_complex",
    f"[0:a]volume={CAT_GAIN}[cat];[1:a]volume={DOG_GAIN}[dog];"
    "[cat][dog]amix=inputs=2:duration=longest:normalize=0,"
    "alimiter=limit=0.95[mix]",
    "-map", "[mix]", "-c:a", "pcm_s16le", str(LAYER_MIX_WAV)
])
print("Layer mix saved:", LAYER_MIX_WAV)

## 10 — Compare three cat options while keeping the dog layer fixed

This is the product-style choice step. We keep the same five-second video, cat prompt, dog control, and mix levels. Only the cat seed changes. If the existing seed-42 cat output used the current prompt, the notebook reuses it; otherwise it regenerates it.

Each option is played alone and then as a video-synchronized mix with the same dog layer. The source video is used for preview only and is not added to the download package.

In [ ]:
# Keep the prompt, video window, and dog layer fixed; vary only the cat seed.
CAT_VARIANT_SEEDS = [42, 137, 2026]
CAT_VARIANTS = []

cat_prompt = PROMPTS["cat_layer"]
if "meow" not in cat_prompt.lower():
    raise ValueError("Run the updated PROMPTS cell with the explicit cat-meow prompt first.")
if not DOG_WAV.is_file():
    raise FileNotFoundError("dog_layer.wav is missing; generate the dog control first.")

for seed in CAT_VARIANT_SEEDS:
    existing = next(
        (
            item for item in CANDIDATES
            if item["id"] == "cat_layer"
            and item["seed"] == seed
            and item["prompt"] == cat_prompt
            and Path(item["wav"]).is_file()
        ),
        None,
    )

    if existing:
        cat_wav = Path(existing["wav"])
        generation_seconds = existing.get("generation_seconds", 0)
        print(f"Reusing matching cat output for seed {seed}.")
    else:
        output_mp4 = OUTPUT_DIR / f"cat_layer_seed_{seed}.mp4"
        cat_wav = OUTPUT_DIR / f"cat_layer_seed_{seed}.wav"
        command = [
            "just", "generate", cat_prompt,
            "--config", str(DEVICE_CONFIG),
            "--attention-engine", "sdpa",
            "--video-file", str(SILENT_WINDOW),
            "--time-length", str(int(WINDOW_SECONDS)),
            "--out", str(output_mp4),
            "--seed", str(seed),
        ]

        print(f"Generating cat option with seed {seed}...")
        started = time.time()
        run(command, cwd=REPO, tail=3000)
        generation_seconds = round(time.time() - started, 2)

        run([
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100",
            "-c:a", "pcm_s16le", str(cat_wav),
        ])

    mix_wav = OUTPUT_DIR / f"cat_{seed}_plus_fixed_dog.wav"
    preview_mp4 = OUTPUT_DIR / f"cat_{seed}_plus_fixed_dog_on_video.mp4"

    run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(cat_wav), "-i", str(DOG_WAV),
        "-filter_complex",
        f"[0:a]volume={CAT_GAIN}[cat];[1:a]volume={DOG_GAIN}[dog];"
        "[cat][dog]amix=inputs=2:duration=longest:normalize=0,"
        "alimiter=limit=0.95[mix]",
        "-map", "[mix]", "-c:a", "pcm_s16le", str(mix_wav),
    ])

    mux = subprocess.run(
        [
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-i", str(SILENT_WINDOW), "-i", str(mix_wav),
            "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
            "-t", str(WINDOW_SECONDS), "-shortest", str(preview_mp4),
        ],
        capture_output=True, text=True,
    )
    if mux.returncode:
        raise RuntimeError(mux.stderr[-3000:])

    option = {
        "id": f"cat_seed_{seed}",
        "seed": seed,
        "prompt": cat_prompt,
        "generation_seconds": generation_seconds,
        "wav": cat_wav,
        "wav_sha256": sha256_file(cat_wav),
        "mix_wav": mix_wav,
        "mix_sha256": sha256_file(mix_wav),
        "preview_mp4": preview_mp4,
    }
    CAT_VARIANTS.append(option)

    display(Markdown(f"### Cat option — seed {seed} — listen to the cat layer"))
    display(Audio(filename=str(cat_wav)))
    display(Markdown(f"### Same option mixed with the fixed dog layer — seed {seed}"))
    show_video_preview(preview_mp4)

print("Cat options are ready. Compare naturalness, cat identity, timing, and dog-layer leakage.")

## 10 — Listen to the outputs

Compare the full-scene generation with each candidate layer and the mix. Focus on whether the cat and dog are distinguishable, whether each layer leaks the other animal, whether timing follows the video, and whether the mix is more useful than the full-scene output.

In [ ]:
display(Markdown("### Selected silent video window"))
show_video_preview(SILENT_WINDOW)

candidate_by_id = {item["id"]: item for item in CANDIDATES}
for candidate_id in ("full_scene", "cat_layer", "dog_layer"):
    item = candidate_by_id[candidate_id]
    display(Markdown(f"### {candidate_id} — {item['generation_seconds']:.0f}s generation"))
    display(Audio(filename=str(item["wav"])))

display(Markdown("### Cat + dog layer mix"))
display(Audio(filename=str(LAYER_MIX_WAV)))

## 11 — Save a private report and download the audio package

The package contains generated audio and a report. It excludes the uploaded source video and generated MP4s, so the source footage is not redistributed by this notebook. The report records the local filename and SHA-256 for traceability.

In [ ]:
REPORT = {
    "experiment": "kandinsky_v2a_cat_dog_event_layers_personal_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": "one-off private generation trial; no training or fine-tuning",
    "source_filename": SOURCE_VIDEO.name,
    "source_sha256": SOURCE_SHA256,
    "source_duration_seconds": SOURCE_DURATION,
    "source_video_streams": VIDEO_STREAMS,
    "source_audio_streams": AUDIO_STREAMS,
    "source_page_url": "",
    "qwen_model": QWEN_ID,
    "qwen_event_text": QWEN_EVENT_TEXT,
    "qwen_event_plan": QWEN_PLAN,
    "qwen_analysis_seconds": QWEN_ANALYSIS_SECONDS,
    "selected_window_start_seconds": WINDOW_START_SECONDS,
    "window_seconds": WINDOW_SECONDS,
    "window_sha256": sha256_file(SILENT_WINDOW),
    "kandinsky_source_commit": SOURCE_COMMIT,
    "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-5s-Diffusers",
    "gpu": gpu.stdout.strip(),
    "seed": SEED,
    "prompts": PROMPTS,
    "candidates": [
        {
            "id": item["id"],
            "prompt": item["prompt"],
            "generation_seconds": item["generation_seconds"],
            "wav": item["wav"].name,
            "wav_sha256": item["wav_sha256"],
            "scores": {
                "scene_match_1_to_5": None,
                "timing_1_to_5": None,
                "quality_1_to_5": None,
                "layer_leakage_1_to_5": None,
                "keep": None,
                "notes": ""
            }
        }
        for item in CANDIDATES
    ],
    "cat_gain_in_mix": CAT_GAIN,
    "dog_gain_in_mix": DOG_GAIN,
    "cat_variant_options": [
        {
            "id": item["id"],
            "seed": item["seed"],
            "prompt": item["prompt"],
            "generation_seconds": item["generation_seconds"],
            "wav": item["wav"].name,
            "wav_sha256": item["wav_sha256"],
            "mix_wav": item["mix_wav"].name,
            "mix_sha256": item["mix_sha256"],
            "scores": {
                "cat_identity_1_to_5": None,
                "naturalness_1_to_5": None,
                "sync_1_to_5": None,
                "dog_leakage_1_to_5": None,
                "keep": None,
                "notes": ""
            }
        }
        for item in CAT_VARIANTS
    ],
    "layer_mix": LAYER_MIX_WAV.name,
    "notes": "Candidate layers are not guaranteed isolated stems."
}

REPORT_PATH = WORK / "pet_event_layers_report.json"
REPORT_PATH.write_text(
    json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8"
)

ZIP_PATH = Path("/content/kandinsky_v2a_pet_event_layers_personal.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT_PATH, "pet_event_layers_report.json")
    for item in CANDIDATES:
        archive.write(item["wav"], "audio/" + item["wav"].name)
    archive.write(LAYER_MIX_WAV, "audio/" + LAYER_MIX_WAV.name)
    for option in CAT_VARIANTS:
        archive.write(option["wav"], "cat_options/" + option["wav"].name)
        archive.write(option["mix_wav"], "cat_options/" + option["mix_wav"].name)

print("Private audio package:", ZIP_PATH, "|", round(ZIP_PATH.stat().st_size / 1e6, 1), "MB")
files.download(str(ZIP_PATH))